# Layer A FAISS インデックス再構築 (Colab GPU 版)

**目的**: `data/staging/layer_a.index` および `layer_a_meta.json` を
`intfloat/multilingual-e5-large` (dim=1024) で再構築する。

**前提**:
- ランタイムを **GPU (T4 以上)** に設定してください。
- Google Drive に `AI_TradeManagement/` リポジトリをマウントするか、
  本ノートブック内で `fefta_law_v5.json` / `ccl_eccn_entries_v8.json` を直接アップロードしてください。

**出力ファイル**:
- `layer_a.index` — FAISS IndexFlatIP (ntotal ≈ 2,200+)
- `layer_a_meta.json` — faiss_id → レコードメタデータ対応表

---

## Step 0: GPU 確認

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠ GPU が利用できません。ランタイム → ランタイムのタイプを変更 → T4 GPU に変更してください。')

## Step 1: ライブラリインストール

In [ ]:
%%capture
!pip install sentence-transformers faiss-gpu numpy

## Step 2: Google Drive マウント（任意）

Drive に `AI_TradeManagement/data/staging/` があればここでマウントします。
なければ Step 3 でファイルを直接アップロードしてください。

In [ ]:
USE_DRIVE = False  # Google Drive を使う場合は True に変更

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STAGING_DIR = '/content/drive/MyDrive/AI_TradeManagement/data/staging'
else:
    STAGING_DIR = '/content/staging'
    import os
    os.makedirs(STAGING_DIR, exist_ok=True)
    print(f'ステージングディレクトリ: {STAGING_DIR}')
    print('次のセルでファイルをアップロードしてください。')

## Step 3: ソースファイルアップロード（Drive を使わない場合）

以下の 2 ファイルをアップロードしてください:
- `fefta_law_v5.json`
- `ccl_eccn_entries_v8.json`

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    import shutil
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = f'{STAGING_DIR}/{fname}'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'保存: {dest} ({len(data):,} bytes)')

## Step 4: データ読み込み

In [ ]:
import json
from pathlib import Path

staging = Path(STAGING_DIR)
PASSAGE_PREFIX = 'passage: '


def embed_text_fefta(rec):
    text = (rec.get('full_text') or rec.get('definition_text') or '').strip()
    if not text:
        return ''
    base = PASSAGE_PREFIX + text
    if rec.get('source_type') == 'parameter':
        val  = rec.get('value_mm') or rec.get('value_numeric') or ''
        unit = rec.get('value_unit') or ''
        if val:
            base += f' [{val}{unit}]'
    return base


def embed_text_eccn(rec):
    parts = []
    eccn  = rec.get('eccn') or rec.get('entry_number') or ''
    title = (rec.get('title') or '').strip()
    desc  = (rec.get('description') or rec.get('full_text') or '').strip()
    if eccn:  parts.append(eccn)
    if title: parts.append(title)
    if desc:  parts.append(desc)
    return PASSAGE_PREFIX + ' '.join(parts) if parts else ''


records = []
type_counts = {}

# 外為法省令
fefta_path = staging / 'fefta_law_v5.json'
if fefta_path.exists():
    with open(fefta_path, encoding='utf-8') as f:
        fefta_data = json.load(f)
    for r in fefta_data.get('records', []):
        et = embed_text_fefta(r)
        if not et:
            continue
        src = r.get('source_type', 'law')
        type_counts[src] = type_counts.get(src, 0) + 1
        records.append({
            'source_type': src,
            'source_name': r.get('source_name', 'fefta_shorei'),
            'article_no':  r.get('article_no', ''),
            'title':       r.get('title', ''),
            'item_no':     r.get('item_no', ''),
            'item_label':  r.get('item_label', ''),
            'chunk_level': r.get('chunk_level', ''),
            'value_mm':    r.get('value_mm'),
            'value_unit':  r.get('value_unit'),
            'full_text':   (r.get('full_text') or r.get('definition_text') or '').strip(),
            'embed_text':  et,
        })
    print(f'FEFTA records loaded: {len([r for r in records if r["source_type"] != "eccn"])}')

# US EAR CCL
eccn_path = staging / 'ccl_eccn_entries_v8.json'
if eccn_path.exists():
    with open(eccn_path, encoding='utf-8') as f:
        eccn_data = json.load(f)
    for r in eccn_data.get('entries', []):
        et = embed_text_eccn(r)
        if not et:
            continue
        type_counts['eccn'] = type_counts.get('eccn', 0) + 1
        records.append({
            'source_type': 'eccn',
            'source_name': 'ccl_ear',
            'article_no':  r.get('eccn') or r.get('entry_number') or '',
            'title':       (r.get('title') or '').strip(),
            'item_no':     r.get('eccn') or r.get('entry_number') or '',
            'item_label':  (r.get('category') or '').strip(),
            'chunk_level': 'entry',
            'value_mm':    None,
            'value_unit':  None,
            'full_text':   (r.get('description') or r.get('full_text') or '').strip(),
            'embed_text':  et,
        })
    print(f'ECCN entries loaded: {type_counts.get("eccn", 0)}')

print(f'\n合計 build records: {len(records)}')
print('source_type breakdown:', type_counts)

# サンプル確認
for r in records[:2]:
    print(f'  [{r["source_type"]}] {r["embed_text"][:100]}')

## Step 5: エンコード (GPU 使用)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME  = 'intfloat/multilingual-e5-large'
BATCH_SIZE  = 64  # GPU T4: 64 で安定。メモリ不足なら 32 に下げてください。

print(f'Loading {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME)
dim = model.get_sentence_embedding_dimension()
print(f'Model ready  dim={dim}')

texts = [r['embed_text'] for r in records]
print(f'\nEncoding {len(texts)} texts (batch_size={BATCH_SIZE}) ...')

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=BATCH_SIZE,
)
emb_matrix = np.asarray(embeddings, dtype='float32')
print(f'Embedding matrix: {emb_matrix.shape}')

## Step 6: FAISS インデックス構築

In [ ]:
import faiss

index = faiss.IndexFlatIP(dim)
index.add(emb_matrix)
print(f'FAISS index built: ntotal={index.ntotal}')

# faiss_id を付与
for i, r in enumerate(records):
    r['faiss_id'] = i

## Step 7: 検索テスト

In [ ]:
def search(query: str, top_k: int = 5):
    q_emb = model.encode([f'query: {query}'], normalize_embeddings=True)
    q_arr = np.asarray(q_emb, dtype='float32')
    scores, ids = index.search(q_arr, top_k)
    for score, idx in zip(scores[0], ids[0]):
        r = records[idx]
        print(f'  [{score:.4f}] [{r["source_type"]}] {r["embed_text"][:100]}')

print('=== Test: 炭素繊維 ===')
search('炭素繊維 複合材料')
print()
print('=== Test: encryption software ===')
search('encryption software export control')
print()
print('=== Test: 遠心分離機 ===')
search('遠心分離機 ウラン濃縮')

## Step 8: 保存

In [ ]:
from datetime import datetime
import os

OUT_INDEX = staging / 'layer_a.index'
OUT_META  = staging / 'layer_a_meta.json'

faiss.write_index(index, str(OUT_INDEX))
print(f'Saved index: {OUT_INDEX}  ({os.path.getsize(OUT_INDEX)/1e6:.1f} MB)')

meta = {
    'total':            index.ntotal,
    'dim':              dim,
    'model':            MODEL_NAME,
    'built_at':         datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'source_breakdown': type_counts,
    'records':          records,
}
with open(OUT_META, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'Saved meta:  {OUT_META}  ({os.path.getsize(OUT_META)/1e6:.1f} MB)')
print(f'Done. ntotal={index.ntotal}  breakdown={type_counts}')

## Step 9: ダウンロード

Google Drive を使っていない場合は、ここでファイルをダウンロードしてください。
ダウンロード後、プロジェクトの `data/staging/` に配置してください。

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    print('layer_a.index をダウンロード中...')
    files.download(str(OUT_INDEX))
    print('layer_a_meta.json をダウンロード中...')
    files.download(str(OUT_META))
    print('完了。data/staging/ に配置してください。')
else:
    print(f'Google Drive に保存済み: {OUT_INDEX}')
    print(f'Google Drive に保存済み: {OUT_META}')